In [18]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import requests
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException, NoSuchElementException, ElementClickInterceptedException  # Import ElementClickInterceptedException
import time

In [19]:
from selenium.common.exceptions import StaleElementReferenceException
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
# URL to scrape
url = 'https://stocktrack.ca/?s=ikea&search=artificial%20plants'

# Initialize the Chrome WebDriver
driver = webdriver.Chrome()

# Navigate to the provided URL
driver.get(url)

# Wait for the iframe to load and switch to it
try:
    wait = WebDriverWait(driver, 50)  # Set a timeout of 40 seconds
    # Locate the iframe by its tag name and switch to it
    iframe = wait.until(EC.presence_of_element_located((By.TAG_NAME, 'iframe')))
    driver.switch_to.frame(iframe)
except TimeoutException:
    print("No iframe found or timed out waiting for iframe to load.")
except NoSuchElementException:
    print("No iframe element found.")


start_dhx_f_id = 1
scraped_products = []

while True:
    try:
        # Wait for the products to load on the current page
        WebDriverWait(driver, 60).until(EC.presence_of_all_elements_located((By.CLASS_NAME, "dhx_list_item")))
        
#         # Initialize a variable to track whether there are elements with dhx_f_id on the current page
        elements_found = False

        # Scrape the products on the current page
        for i in range(start_dhx_f_id, start_dhx_f_id + 5):
            div_xpath = f'//div[@dhx_f_id="{i}"]'

            for _ in range(3):
                try:
                    div_to_click = driver.find_element(By.XPATH, div_xpath)
                    if div_to_click:
                        div_to_click.click()
                    else:
                        break
                    div_element = driver.find_element(By.XPATH, f'//div[@dhx_f_id="{i}"]')

                    # Extract product information from the div element
                    image = div_element.find_element(By.TAG_NAME, 'img').get_attribute('src')
                    product_name = div_element.find_element(By.TAG_NAME, 'a').text
                    product_link = div_element.find_element(By.TAG_NAME, 'a').get_attribute('href')

                    # Split the text by line breaks to extract individual pieces of information
                    # Extract the text content from the parent element
                    product_info = div_element.text

                    # Split the text into lines and extract the relevant information
                    lines = product_info.split('\n')

                    # Initialize variables to store extracted information
                    sku = None
                    size = None
                    price = None

                    # Iterate through the lines to find relevant information
                    for line in lines:
                        if line.startswith("SKU:"):
                            sku = line.replace("SKU:", "").strip()
                        elif "cm" in line:
                            size = line.strip()
                        elif line.startswith("Price:"):
                            price = line.replace("Price:", "").strip()
                            
                    # Split the price into old and new price if applicable
                    if " " in price:
                        prices = price.split()
                        old_price = prices[0]  # First part is old price
                        new_price = prices[1]  # Second part is new price
                    else:
                        old_price = price
                        new_price = 'N/A'                        
                    # Print the extracted information
                    print(i)
                    print("image:", image)
                    print("Product Name :", product_name)
                    print("Product Link :", product_link)
                    print("SKU:", sku)
                    print("Size:", size)
                    print("Old Price:", old_price)
                    print("New Price:", new_price)
                    try:
                        stock_number_element = WebDriverWait(driver, 25).until(
                        EC.presence_of_element_located((By.XPATH, "//td[contains(text(), 'Coquitlam')]/following-sibling::td[3]"))
                        )
                        stock_number_coq = stock_number_element.text
                        print("Coquitlam Store Stock Number:", stock_number_coq)
                        rich_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Richmond')]/following-sibling::td[3]")
                        stock_number_rich = rich_element.text
                        print("Richmond Store Stock Number:", stock_number_rich)
                    except TimeoutException:
                        print("Timed out waiting for the Coquitlam stock number to load")
                    except NoSuchElementException:
                        print("Coquitlam stock number element not found.")       
                    print()
                    product_data = {
                    'image_url': image,
                    'product_name': product_name,
                    'product_link': product_link,
                    'product_size' : size,
                    'product_sku': sku,
                    'product_price_old': old_price,
                    'product_price_new' : new_price,
                    'stock_number_coquitlam' : stock_number_coq,
                    'stock_number_richmond' : stock_number_rich
                    }
                    # Append the product data to the list
                    scraped_products.append(product_data)
                    # Set elements_found to True since elements with dhx_f_id were found
                    elements_found = True

                    break  # Exit the loop if the click is successful
                except StaleElementReferenceException:
                    continue  # Retry if a StaleElementReferenceException occurs
                except ElementClickInterceptedException:
                    print("Element click intercepted. Trying again.")
        
        # If no elements with dhx_f_id were found on the current page, break out of the loop
        if not elements_found:
            break

        print()
        
        # Check if the "Next" button is clickable
        next_page_link = driver.find_element(By.XPATH, "//div[@dhx_p_id='next']")
        if not next_page_link.is_enabled():
            break  # Break out of the loop if the "Next" button is not clickable

        # Move to the next page by clicking the 'next page' link
        ActionChains(driver).move_to_element(next_page_link).click(next_page_link).perform()
        start_dhx_f_id += 5
    except TimeoutException:
        print("Timed out waiting for products to load.")
    
    except NoSuchElementException:
            print(f"Element with dhx_f_id='{i}' not found. Exiting loop.")
            break  # Exit the loop if the element is not found



# Close the WebDriver
# Print the scraped product data

time.sleep(10)
driver.quit()





1
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-bamboo__0748884_pe745273_s5.jpg
Product Name FEJKA, Artificial potted plant
Product Link https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-bamboo-10467804/
SKU: 10467804
Size: 23 cm (9 ")
Old Price: $69.99
New Price: N/A
Coquitlam Store Stock Number: 19
Richmond Store Stock Number: 13

2
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-plant-with-wall-holder-indoor-outdoor-green-lilac__1184665_pe898020_s5.jpg
Product Name FEJKA, Artificial plant with wall holder
Product Link https://www.ikea.com/ca/en/p/fejka-artificial-plant-with-wall-holder-indoor-outdoor-green-lilac-30548625/
SKU: 30548625
Size: None
Old Price: $6.99
New Price: N/A
Coquitlam Store Stock Number: 51
Richmond Store Stock Number: 155

3
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-weeping-fig__0748885_pe745274_s5.jpg
Product Name FEJK

Coquitlam Store Stock Number: 101
Richmond Store Stock Number: 66


21
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-leaf-eucalyptus-green__0638910_pe699263_s5.jpg
Product Name SMYCKA, Artificial leaf
Product Link https://www.ikea.com/ca/en/p/smycka-artificial-leaf-eucalyptus-green-80335773/
SKU: 80335773
Size: 65 cm (25 ½ ")
Old Price: $7.99
New Price: $5.99
Coquitlam Store Stock Number: 884
Richmond Store Stock Number: 51

22
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-bouquet-protea-orange-brown__1188129_pe899363_s5.jpg
Product Name SMYCKA, Artificial bouquet
Product Link https://www.ikea.com/ca/en/p/smycka-artificial-bouquet-protea-orange-brown-10559988/
SKU: 10559988
Size: 50 cm (19 ¾ ")
Old Price: $9.99
New Price: N/A
Timed out waiting for the Coquitlam stock number to load

23
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-plant-wall-mounted-indoor-outdoor-green__1183258_pe897457_s5.jpg
Product Name FEJKA, Artific

Coquitlam Store Stock Number: 386
Richmond Store Stock Number: 180


41
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-house-bamboo__0711609_pe728351_s5.jpg
Product Name FEJKA, Artificial potted plant
Product Link https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-house-bamboo-60433939/
SKU: 60433939
Size: 9 cm (3 ½ ")
Old Price: $4.99
New Price: N/A
Coquitlam Store Stock Number: 144
Richmond Store Stock Number: 178

42
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-eucalyptus__0674947_pe718059_s5.jpg
Product Name FEJKA, Artificial potted plant
Product Link https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-eucalyptus-40452368/
SKU: 40452368
Size: 15 cm (6 ")
Old Price: $24.99
New Price: N/A
Coquitlam Store Stock Number: 29
Richmond Store Stock Number: 0

43
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-wreath-green__0641805

Timed out waiting for the Coquitlam stock number to load

62
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-rose-pink__0614177_pe686803_s5.jpg
Product Name FEJKA, Artificial potted plant
Product Link https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-rose-pink-00395313/
SKU: 00395313
Size: 9 cm (3 ½ ")
Old Price: $4.99
New Price: N/A
Coquitlam Store Stock Number: 27
Richmond Store Stock Number: 16

63
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-fern__0522691_pe643398_s5.jpg
Product Name FEJKA, Artificial potted plant
Product Link https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-fern-30433945/
SKU: 30433945
Size: 9 cm (3 ½ ")
Old Price: $9.99
New Price: N/A
Coquitlam Store Stock Number: 71
Richmond Store Stock Number: 62

64
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-flower-cherry-blossoms-pink__0611007_pe685254_s5.jp

Timed out waiting for the Coquitlam stock number to load

83
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-flower-indoor-outdoor-poppy-pink__1188145_pe899373_s5.jpg
Product Name SMYCKA, Artificial flower
Product Link https://www.ikea.com/ca/en/p/smycka-artificial-flower-indoor-outdoor-poppy-pink-30560151/
SKU: 30560151
Size: 27 cm (10 ¾ ")
Old Price: $1.49
New Price: N/A
Coquitlam Store Stock Number: 60
Richmond Store Stock Number: 23

84
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-flower-peony-white__0611399_pe685423_s5.jpg
Product Name SMYCKA, Artificial flower
Product Link https://www.ikea.com/ca/en/p/smycka-artificial-flower-peony-white-80409783/
SKU: 80409783
Size: 30 cm (11 ¾ ")
Old Price: $1.99
New Price: N/A
Coquitlam Store Stock Number: 267
Richmond Store Stock Number: 399

85
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-mosaic-plant-hanging__1248044_pe922954_s5.jpg
Product Name FEJ

Coquitlam Store Stock Number: 41
Richmond Store Stock Number: 36

103
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-tulip-pink__1248037_pe922950_s5.jpg
Product Name FEJKA, Artificial potted plant
Product Link https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-tulip-pink-60571681/
SKU: 60571681
Size: 9 cm (3 ½ ")
Old Price: $4.99
New Price: N/A
Coquitlam Store Stock Number: 42
Richmond Store Stock Number: 18

104
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-flower-indoor-outdoor-camellia-red__1248066_pe922970_s5.jpg
Product Name SMYCKA, Artificial flower
Product Link https://www.ikea.com/ca/en/p/smycka-artificial-flower-indoor-outdoor-camellia-red-50571790/
SKU: 50571790
Size: 28 cm (11 ")
Old Price: $1.49
New Price: N/A
Coquitlam Store Stock Number: 251
Richmond Store Stock Number: 253

105
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-bouquet-indoor-outdoor-yellow-orang

In [20]:
for product in scraped_products:
    print(product)
    print() 

{'image_url': 'https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-bamboo__0748884_pe745273_s5.jpg', 'product_name': 'FEJKA, Artificial potted plant', 'product_link': 'https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-bamboo-10467804/', 'product_size': '23 cm (9 ")', 'product_sku': '10467804', 'product_price_old': '$69.99', 'product_price_new': 'N/A', 'stock_number_coquitlam': '19', 'stock_number_richmond': '13'}

{'image_url': 'https://www.ikea.com/ca/en/images/products/fejka-artificial-plant-with-wall-holder-indoor-outdoor-green-lilac__1184665_pe898020_s5.jpg', 'product_name': 'FEJKA, Artificial plant with wall holder', 'product_link': 'https://www.ikea.com/ca/en/p/fejka-artificial-plant-with-wall-holder-indoor-outdoor-green-lilac-30548625/', 'product_size': None, 'product_sku': '30548625', 'product_price_old': '$6.99', 'product_price_new': 'N/A', 'stock_number_coquitlam': '51', 'stock_number_richmond': '155'}

{'image_url':

In [24]:
import csv
csv_file_path = "products_artifical_plants_2024_01_24.csv"
# Create or open the CSV file for writing
with open(csv_file_path, mode='w', newline='') as csv_file:
    # Define the CSV headers (column names)
    fieldnames = [
        'image_url',
        'product_name',
        'product_link',
        'product_size',
        'product_sku',
        'product_price_old',
        'product_price_new',
        'stock_number_coquitlam',
        'stock_number_richmond'
    ]

    # Create a CSV writer
    csv_writer = csv.DictWriter(csv_file, fieldnames=fieldnames)

    # Write the header row to the CSV file
    csv_writer.writeheader()

    # Iterate through the scraped_products list and write each product's data
    for product in scraped_products:
        csv_writer.writerow(product)

print(f"CSV file has been created : '{csv_file_path}'.")

Data has been written to 'products_artifical_plants_2024_01_24.csv'.


In [21]:
# Loop through the pages using the 'next page' button
# while True:
#     try:
#         # Wait for the 'next page' button to be clickable
#         next_page_button = WebDriverWait(driver, 10).until(
#             EC.element_to_be_clickable((By.XPATH, "//div[@dhx_p_id='next']"))
#         )
        
#         # Click the 'next page' button
#         next_page_button.click()


#         WebDriverWait(driver, 10).until(
#             EC.staleness_of(next_page_button)
#         )
#         time.sleep(5)
#     except (TimeoutException, NoSuchElementException):
#         # Break the loop if 'next page' button is not found or not clickable
#         break

In [22]:
# for i in range(6, 11):
#     div_xpath = f'//div[@dhx_f_id="{i}"]'
    
#     for _ in range(3):
#         try:
#             div_to_click = driver.find_element(By.XPATH, div_xpath)
#             div_to_click.click()
#             clicked_element_html = div_to_click.get_attribute("outerHTML")
#             div_element = driver.find_element(By.XPATH, f'//div[@dhx_f_id="{i}"]')

#             # Extract product information from the div element
#             image = div_element.find_element(By.TAG_NAME, 'img').get_attribute('src')
#             product_name = div_element.find_element(By.TAG_NAME, 'a').text
#             product_link = div_element.find_element(By.TAG_NAME, 'a').get_attribute('href')

#             # Split the text by line breaks to extract individual pieces of information
#             # Extract the text content from the parent element
#             product_info = div_element.text

#             # Split the text into lines and extract the relevant information
#             lines = product_info.split('\n')

#             # Initialize variables to store extracted information
#             sku = None
#             size = None
#             price = None

#             # Iterate through the lines to find relevant information
#             for line in lines:
#                 if line.startswith("SKU:"):
#                     sku = line.replace("SKU:", "").strip()
#                 elif "cm" in line:
#                     size = line.strip()
#                 elif line.startswith("Price:"):
#                     price = line.replace("Price:", "").strip()

#             # Print the extracted information
#             print("SKU:", sku)
#             print("Size:", size)
#             print("Price:", price)
#             try:
#                 stock_number_element = WebDriverWait(driver, 5).until(
#                 EC.presence_of_element_located((By.XPATH, "//td[contains(text(), 'Coquitlam')]/following-sibling::td[3]"))
#                 )
#                 stock_number_coq = stock_number_element.text
#                 print("Coquitlam Store Stock Number:", stock_number_coq)
#                 rich_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Richmond')]/following-sibling::td[3]")
#                 stock_number_rich = rich_element.text
#                 print("Richmond Store Stock Number:", stock_number_rich)
#             except TimeoutException:
#                 print("Timed out waiting for the Coquitlam stock number to load")
#             except NoSuchElementException:
#                 print("Coquitlam stock number element not found.") 
#             print()
#             break  # Exit the loop if the click is successful
#         except StaleElementReferenceException:
#             continue  # Retry if a StaleElementReferenceException occurs
  



In [23]:
# i=1
# try:
#     # Wait until all elements with class 'dhx_list_item' are present
#     try:
#         products_list = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "dhx_list_item")))
#     except TimeoutException:        
#         print("Timed out waiting for products to load within the iframe.")
#     # List to store scraped product data
#     scraped_products = []
    
#     for product in products_list:
#         try:
#             # Interact with the 1st product on the page
#             # Extract product details like image, name, and link
#             image = product.find_element(By.TAG_NAME, 'img').get_attribute('src')
#             name = product.find_element(By.TAG_NAME, 'a').text
#             link = product.find_element(By.TAG_NAME, 'a').get_attribute('href')

#             # Extract and process product information like size, SKU, and price     
#             product_info = product.find_elements(By.XPATH, './/td')[1].text
#             product_info_lines = product_info.split('\n')
#             #second_td_element = product.find_elements(By.TAG_NAME, 'td')[1]
#             #product_size = second_td_element.text.strip()  # Extract size information from the second <td> element

#             # Parse the product information based on its format
#             if ":" in product_info_lines[1].strip():
#                 product_size = "N/A"  # Set a default value if size is not available
#                 product_sku = product_info_lines[1].split(': ')[1]
#                 product_price = product_info_lines[2].split(': ')[1]
#             else:
#                 product_size = product_info_lines[1].strip() 
#                 product_sku = product_info_lines[2].split(': ')[1]
#                 product_price = product_info_lines[3].split(': ')[1]   

#             # Split the price into old and new price if applicable
#             if " " in product_price:
#                 prices = product_price.split()
#                 old_price = prices[0]  # First part is old price
#                 new_price = prices[1]  # Second part is new price
#             else:
#                 old_price = product_price
#                 new_price = 'N/A'
#             # Extract product availability status
#             product_status_element = product.find_element(By.CLASS_NAME, 'instock')
#             product_stat = product_status_element.text


# # Store the product data in a dictionary
#             print(i)
#             div_xpath = f'//div[@dhx_f_id="{i}"]'
#             div_to_click = driver.find_element(By.XPATH, div_xpath)
#             div_to_click.click()
#             i +=1


# #                 for _ in range(3):
# #                     try:
# #                         div_to_click = driver.find_element(By.XPATH, div_xpath)
# #                         i+=1
# #                         div_to_click.click()
# #                         break  # Exit the loop if the click is successful
# #                     except StaleElementReferenceException:
# #                         continue  # Retry if a StaleElementReferenceException occurs

#             # Extract stock numbers for Coquitlam and Richmond stores
#             try:
#                 stock_number_element = WebDriverWait(driver, 5).until(
#                 EC.presence_of_element_located((By.XPATH, "//td[contains(text(), 'Coquitlam')]/following-sibling::td[3]"))
#                 )
#                 stock_number_coq = stock_number_element.text
#                 #print("Coquitlam Store Stock Number:", stock_number_coq)
#                 rich_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Richmond')]/following-sibling::td[3]")
#                 stock_number_rich = rich_element.text
#                 #print("Richmond Store Stock Number:", stock_number_rich)

#             except TimeoutException:
#                 print("Timed out waiting for the Coquitlam stock number to load")
#             except NoSuchElementException:
#                 print("Coquitlam stock number element not found.")

#             product_data = {
#                 'image_url': image,
#                 'product_name': name,
#                 'product_link': link,
#                 'product_size' : product_size,
#                 'product_sku': product_sku,
#                 'product_price_old': old_price,
#                 'product_price_new' : new_price,
#                 'product_status': product_stat,
#                 'stock_number_coquitlam' : stock_number_coq,
#                 'stock_number_richmond' : stock_number_rich
#             }
#             # Append the product data to the list
#             scraped_products.append(product_data)
#             # Output the information
#             #print(f"Image: {image}, Name: {name}, Link: {link}, product_size : {product_size}, product_sku : {product_sku}, product_stat : {product_stat}")
#         except StaleElementReferenceException:
#             continue  # Retry this iteration if a StaleElementReferenceException occurs
# except TimeoutException:
#     print("Timed out waiting for products to load")


# for product in scraped_products:
#     print(product)
#     print()        

# try:
#     # Locate the 'next page' link using its attributes
#     next_page_link = driver.find_element(By.XPATH, "//div[@dhx_p_id='next']")

#     # Click on the 'next page' link
#     ActionChains(driver).move_to_element(next_page_link).click(next_page_link).perform()

#     # Wait for the new page to load (you can use a WebDriverWait)
#     try:
#         WebDriverWait(driver, 50).until(EC.presence_of_all_elements_located((By.CLASS_NAME, "dhx_list_item")))
#     except TimeoutException:
#         print("Timed out waiting for products to load on the next page.")

#     # Scrape the products on the new page (similar to your existing code)
#     try:
#         products_list = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "dhx_list_item")))

#     except TimeoutException:
#         print("Timed out waiting for products to load on the new page.")

# except NoSuchElementException:
#     print("No 'next page' link found. Exiting the loop.")